# 18 · Rétropropagation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch18_backprop/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch18_backprop/03_notebook.ipynb`, créée par `python tools/start_chapter.py 18`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 18.Q1 à 18.Q12, 18.R1 à 18.R3, 18.1 à 18.5, 18.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 18.12 à 18.17 | les losses et leur gradient, la méthode très lente, la passe arrière d'une couche, gradient_check, une chasse aux gradients faux (*prochaine session de génération*) | | | |
| B | 18.18 à 18.21 | mlp_backward, le réseau 2-4-4-1 à 37 poids sur les lunes, la même expérience avec scikit-learn, les figures du learning rate (*prochaine session de génération*) | | | |
| C | 18.22 à 18.25 | lire des courbes de loss, balayer le learning rate, des neurones saturés et morts sur MNIST, un test pytest de gradient (*prochaine session de génération*) | | | |
| D | 18.26 à 18.29 | le micro-autograd Value, quatre gradients comparés, MiniMLP, le défi MNIST (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")  # your own library

## Objectifs et rappel express

- Calculer à la main les deltas, les gradients et un pas de mise à jour d'un petit réseau, avec et sans activation.
- Écrire et vérifier contre PyTorch les fonctions de `mylearn.nn.backward` (les losses avec leur gradient, `dense_backward`, `mlp_backward`, `gradient_check`) et le micro-autograd `mylearn.autograd_mini` (`Value`, `MiniMLP`).
- Entraîner un MLP from scratch et diagnostiquer le learning rate, les plateaux, la saturation et les neurones morts.

**Rappel express.** $\delta_j = \partial L / \partial z_j$ ; sortie, loss quadratique : $\delta_k = (\hat{y}_k - y_k)\,f'(z_k)$ ; caché : $\delta_j = f'(z_j) \sum_k w_{jk}\,\delta_k$ ; $\partial L / \partial w_{ij} = a_i\,\delta_j$, $\partial L / \partial b_j = \delta_j$ ; $w \leftarrow w - \eta\,\partial L / \partial w$. En matrices : $\boldsymbol{\Delta} = \mathbf{G} \odot f'(\mathbf{Z})$, $d\mathbf{W} = \mathbf{A}_{\text{prev}}^\top\boldsymbol{\Delta}$, $d\mathbf{b} = \mathbf{1}^\top\boldsymbol{\Delta}$, $\mathbf{G}_{\text{prev}} = \boldsymbol{\Delta}\mathbf{W}^\top$. Cross-entropy sur les logits : $(\mathbf{P} - \mathbf{Y})/n$ ; binaire : $(\sigma(\mathbf{z}) - \mathbf{y})/n$. Vérification : $\frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$, erreur relative $\frac{|a - n|}{\max(|a| + |n|, 10^{-12})}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 18.1 à 18.5 et 🧮 18.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un sens : `+1` (monter) ou `-1` (baisser) ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix ou une suite : les lettres collées, dans l'ordre demandé (`"CADB"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 18.6 à 18.8, la réflexion (🗣️ 18.10, 📄 18.11) et l'entretien se corrigent avec `05_solutions.md`.

**18.Q1 — Pourquoi apprendre la rétropropagation si PyTorch la fait pour toi ?**

In [ ]:
# 18.Q1: the values you found on paper
answer_18_Q1a = ...  # a) True or False
answer_18_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_18_Q1a, answer_18_Q1b, answer_18_Q1c]):
    wb.check(f"18.Q1{letter}", answer)

**18.Q2 — Perturber un poids au hasard : pourquoi est-ce si lent ?**

In [ ]:
# 18.Q2: the values you found on paper
answer_18_Q2a = ...  # a) a whole number
answer_18_Q2b = ...  # b) a proportion
answer_18_Q2c = ...  # c) True or False
answer_18_Q2d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_18_Q2a, answer_18_Q2b, answer_18_Q2c, answer_18_Q2d]):
    wb.check(f"18.Q2{letter}", answer)

**18.Q3 — Ce que le gradient dit à chaque poids, et le cas du biais**

In [ ]:
# 18.Q3: the values you found on paper
answer_18_Q3a = ...  # a) +1 (up) or -1 (down)
answer_18_Q3b = ...  # b) +1 (up) or -1 (down)
answer_18_Q3c = ...  # c) True or False
answer_18_Q3d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_18_Q3a, answer_18_Q3b, answer_18_Q3c, answer_18_Q3d]):
    wb.check(f"18.Q3{letter}", answer)

**18.Q4 — Le delta d'un neurone : signe et effet sur l'erreur**

In [ ]:
# 18.Q4: the values you found on paper
answer_18_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q4b = ...  # b) +1 (up) or -1 (down)
answer_18_Q4c = ...  # c) True or False
answer_18_Q4d = ...  # d) a number
answer_18_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q4a, answer_18_Q4b, answer_18_Q4c, answer_18_Q4d, answer_18_Q4e]):
    wb.check(f"18.Q4{letter}", answer)

**18.Q5 — Le réseau minuscule : couches, poids et leurs noms**

In [ ]:
# 18.Q5: the values you found on paper
answer_18_Q5a = ...  # a) a whole number
answer_18_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q5c = ...  # c) a whole number
answer_18_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_18_Q5e = ...  # e) a list of 4 numbers

for letter, answer in zip("abcde", [answer_18_Q5a, answer_18_Q5b, answer_18_Q5c, answer_18_Q5d, answer_18_Q5e]):
    wb.check(f"18.Q5{letter}", answer)

**18.Q6 — Delta de sortie, delta caché, mise à jour d'un poids**

In [ ]:
# 18.Q6: the values you found on paper
answer_18_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q6b = ...  # b) a number
answer_18_Q6c = ...  # c) a number
answer_18_Q6d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q6a, answer_18_Q6b, answer_18_Q6c, answer_18_Q6d]):
    wb.check(f"18.Q6{letter}", answer)

**18.Q7 — Rétropropager couche par couche, puis avec une activation**

In [ ]:
# 18.Q7: the values you found on paper
answer_18_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q7b = ...  # b) a whole number
answer_18_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_18_Q7d = ...  # d) a number
answer_18_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q7a, answer_18_Q7b, answer_18_Q7c, answer_18_Q7d, answer_18_Q7e]):
    wb.check(f"18.Q7{letter}", answer)

**18.Q8 — Learning rate trop grand ou trop petit : reconnaître les symptômes**

In [ ]:
# 18.Q8: the values you found on paper
answer_18_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q8c = ...  # c) True or False
answer_18_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_18_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q8a, answer_18_Q8b, answer_18_Q8c, answer_18_Q8d, answer_18_Q8e]):
    wb.check(f"18.Q8{letter}", answer)

**18.Q9 — Ce que la rétropropagation propage vraiment**

In [ ]:
# 18.Q9: the values you found on paper
answer_18_Q9a = ...  # a) True or False
answer_18_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q9c = ...  # c) four letters in order, e.g. "CADB"
answer_18_Q9d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q9a, answer_18_Q9b, answer_18_Q9c, answer_18_Q9d]):
    wb.check(f"18.Q9{letter}", answer)

**18.Q10 — Neurones saturés et ReLU mortes**

In [ ]:
# 18.Q10: the values you found on paper
answer_18_Q10a = ...  # a) True or False
answer_18_Q10b = ...  # b) a number
answer_18_Q10c = ...  # c) letters glued in alphabetical order, e.g. "BD"
answer_18_Q10d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q10a, answer_18_Q10b, answer_18_Q10c, answer_18_Q10d]):
    wb.check(f"18.Q10{letter}", answer)

**18.Q11 — Mini-batches, mises à jour parallèles et coût du backward**

In [ ]:
# 18.Q11: the values you found on paper
answer_18_Q11a = ...  # a) a whole number
answer_18_Q11b = ...  # b) True or False
answer_18_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_18_Q11d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q11a, answer_18_Q11b, answer_18_Q11c, answer_18_Q11d]):
    wb.check(f"18.Q11{letter}", answer)

**18.Q12 — Aucune garantie, une longue histoire, des équations à apprivoiser**

In [ ]:
# 18.Q12: the values you found on paper
answer_18_Q12a = ...  # a) True or False
answer_18_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q12c = ...  # c) a year
answer_18_Q12d = ...  # d) a year
answer_18_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_18_Q12a, answer_18_Q12b, answer_18_Q12c, answer_18_Q12d, answer_18_Q12e]):
    wb.check(f"18.Q12{letter}", answer)

**18.R1 — La dérivée de la sigmoïde à partir de sa sortie**

In [ ]:
# 18.R1: the values you found on paper
answer_18_R1a = ...  # a) a number
answer_18_R1b = ...  # b) a number
answer_18_R1c = ...  # c) a number
answer_18_R1d = ...  # d) True or False
answer_18_R1e = ...  # e) a number

for letter, answer in zip("abcde", [answer_18_R1a, answer_18_R1b, answer_18_R1c, answer_18_R1d, answer_18_R1e]):
    wb.check(f"18.R1{letter}", answer)

**18.R2 — Chercher le meilleur learning rate par recherche en grille**

In [ ]:
# 18.R2: the values you found on paper
answer_18_R2a = ...  # a) a whole number
answer_18_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_18_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_18_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_R2a, answer_18_R2b, answer_18_R2c, answer_18_R2d]):
    wb.check(f"18.R2{letter}", answer)

**18.R3 — Représentation, évaluation, optimisation dans un MLP**

In [ ]:
# 18.R3: the values you found on paper
answer_18_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_18_R3b = ...  # b) the letter of your choice, e.g. "E"
answer_18_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_18_R3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_R3a, answer_18_R3b, answer_18_R3c, answer_18_R3d]):
    wb.check(f"18.R3{letter}", answer)

**Ex 18.1 — Le delta, amplificateur de changement**

In [ ]:
# Ex 18.1: the values you found on paper
answer_18_1a = ...  # a) a number
answer_18_1b = ...  # b) a number
answer_18_1c = ...  # c) a number
answer_18_1d = ...  # d) a number
answer_18_1e = ...  # e) a number
answer_18_1f = ...  # f) a number with 2 decimals
answer_18_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_18_1a, answer_18_1b, answer_18_1c, answer_18_1d, answer_18_1e, answer_18_1f, answer_18_1g]):
    wb.check(f"18.1{letter}", answer)

**Ex 18.2 — Deltas de sortie avec l'erreur quadratique : attention au signe**

In [ ]:
# Ex 18.2: the values you found on paper
answer_18_2a = ...  # a) a number
answer_18_2b = ...  # b) a number
answer_18_2c = ...  # c) a number
answer_18_2d = ...  # d) a number
answer_18_2e = ...  # e) a number
answer_18_2f = ...  # f) a number
answer_18_2g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_18_2a, answer_18_2b, answer_18_2c, answer_18_2d, answer_18_2e, answer_18_2f, answer_18_2g]):
    wb.check(f"18.2{letter}", answer)

**Ex 18.3 — Descente sur une parabole : quel learning rate ?**

In [ ]:
# Ex 18.3: the values you found on paper
answer_18_3a = ...  # a) a number
answer_18_3b = ...  # b) a number
answer_18_3c = ...  # c) a number with 3 decimals
answer_18_3d = ...  # d) a number
answer_18_3e = ...  # e) a number
answer_18_3f = ...  # f) a number
answer_18_3g = ...  # g) a number
answer_18_3h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_18_3a, answer_18_3b, answer_18_3c, answer_18_3d, answer_18_3e, answer_18_3f, answer_18_3g, answer_18_3h]):
    wb.check(f"18.3{letter}", answer)

**Ex 18.4 — Rétropropagation complète du réseau minuscule, sans activation**

In [ ]:
# Ex 18.4: the values you found on paper
answer_18_4a = ...  # a) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4b = ...  # b) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4c = ...  # c) a number
answer_18_4d = ...  # d) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4e = ...  # e) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4f = ...  # f) a list of 4 numbers
answer_18_4g = ...  # g) a list of 4 numbers
answer_18_4h = ...  # h) a number
answer_18_4i = ...  # i) a number with 4 decimals

for letter, answer in zip("abcdefghi", [answer_18_4a, answer_18_4b, answer_18_4c, answer_18_4d, answer_18_4e, answer_18_4f, answer_18_4g, answer_18_4h, answer_18_4i]):
    wb.check(f"18.4{letter}", answer)

**Ex 18.5 — Le même réseau avec une sigmoïde cachée : le facteur σ'(z)**

In [ ]:
# Ex 18.5: the values you found on paper
answer_18_5a = ...  # a) a number with 4 decimals
answer_18_5b = ...  # b) a number with 4 decimals
answer_18_5c = ...  # c) a number with 4 decimals
answer_18_5d = ...  # d) a number with 4 decimals
answer_18_5e = ...  # e) a number with 4 decimals
answer_18_5f = ...  # f) a number with 4 decimals
answer_18_5g = ...  # g) a number with 4 decimals
answer_18_5h = ...  # h) a number with 4 decimals

for letter, answer in zip("abcdefgh", [answer_18_5a, answer_18_5b, answer_18_5c, answer_18_5d, answer_18_5e, answer_18_5f, answer_18_5g, answer_18_5h]):
    wb.check(f"18.5{letter}", answer)

**Ex 18.9 — Fermi : rétropropagation contre force brute**

In [ ]:
# Ex 18.9: the values you found on paper
answer_18_9a = ...  # a) a whole number
answer_18_9b = ...  # b) a whole number of thousands
answer_18_9c = ...  # c) a whole number
answer_18_9d = ...  # d) a whole number of thousands
answer_18_9e = ...  # e) a whole number of milliseconds
answer_18_9f = ...  # f) a number of hours, 1 decimal

for letter, answer in zip("abcdef", [answer_18_9a, answer_18_9b, answer_18_9c, answer_18_9d, answer_18_9e, answer_18_9f]):
    wb.check(f"18.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dérouler à la main la passe avant, les deltas et un pas de mise à jour d'un petit réseau ?
2. Sais-tu écrire les gradients d'une couche dense pour un mini-batch, et vérifier leurs formes ?
3. Sais-tu vérifier un gradient par différences finies, et reconnaître un learning rate mal réglé ?

**Pour aller plus loin** : le chapitre 2 du livre de M. Nielsen, les notes de CS231n et micrograd d'A. Karpathy, cités dans la fiche. La suite : le ch. 19, où les optimiseurs remplaceront la mise à jour $w \leftarrow w - \eta g$.

*Les exercices de code 18.12 à 18.29 (parties A à D, ta librairie `nn/backward.py` et `autograd_mini.py`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 18` pour obtenir le notebook complet.*